## Import libraries

In [2]:
import pandas as pd
import yfinance as yf
from pathlib import Path
import fredapi
import io
import zipfile
import requests
import config
from fredapi import Fred

## Functions

In [4]:
def validate_yahoo_tickers(tickers):
    invalid_tickers = []

    for ticker in tickers:
        try:
            data = yf.download(
                ticker,
                period="5d",
                interval="1d",
                progress=False
            )

            if data.empty:
                invalid_tickers.append(ticker)

        except Exception:
            invalid_tickers.append(ticker)

    if invalid_tickers:
        print("Invalid Yahoo Finance tickers:")
        for ticker in invalid_tickers:
            print(f"  {ticker}")

In [5]:
def download_yahoo_data(ticker, output_dir):  
    try:
        data = yf.download(
            ticker,
            period="max",
            interval="1d",
            auto_adjust=False,
            actions=True,
            progress=False,
        )

        if data.empty:
            return {
                "Ticker": ticker,
                "Status": "NO DATA"
            }

        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        output_file = output_dir / f"{ticker}.csv"
        data.to_csv(output_file)

        print(f"Downloaded: {ticker}")

    except Exception as e:
        return {
            "Ticker": ticker,
            "Status": f"ERROR: {e}"
        }

In [6]:
def download_fred_series(series_id, fred, output_dir):
    """
    Download a FRED series and save the raw observations.
    """

    try:
        data = fred.get_series(series_id)

        data = pd.DataFrame(data, columns=[series_id])
        data.index.name = "Date"

        if data.empty:
            print(f"WARNING: No data returned for {series_id}")
            return False

        output_file = output_dir / f"{series_id}.csv"
        data.to_csv(output_file)

        print(f"Downloaded: {series_id}")
        return True

    except Exception as e:
        print(f"ERROR downloading {series_id}: {e}")
        return False

In [7]:
def download_fama_french_files(url, filename, output_dir):
    """
    Download a Kenneth French Data Library ZIP file,
    extract the relevant CSV file and save it.
    """

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            files = z.namelist()

            # Extract all files
            z.extractall(output_dir)

        print(f"Downloaded: {files}")
        return True

    except Exception as e:
        print(f"ERROR downloading {filename}: {e}")
        return False

## Yahoo Finance Data

In [9]:
# Directory where raw downloaded data will be stored
YAHOO_RAW_DIR = config.RAW_DATA_DIR / "yahoo"
YAHOO_RAW_DIR.mkdir(parents=True, exist_ok=True)


# Read the master investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Check duplicate tickers
duplicates = universe[universe["Ticker"].duplicated(keep=False)]

if not duplicates.empty:
    print("\nWARNING: Duplicate tickers found:")
    print(duplicates[["Ticker", "Instrument"]])


# Check missing values
missing_values = universe[
    universe.isna().any(axis=1)
]

if not missing_values.empty:
    print("\nWARNING: Missing universe information:")
    display(missing_values)

# Validate Yahoo tickers and download raw data
yahoo_universe = universe[universe["Source"].str.strip().str.lower() == "yahoo finance"].copy()

validate_yahoo_tickers(yahoo_universe["Ticker"])

audit_records = []

for ticker in yahoo_universe["Ticker"]:
    download_yahoo_data(
        ticker=ticker,
        output_dir=YAHOO_RAW_DIR
    )

Downloaded: SPY
Downloaded: VT
Downloaded: QQQ
Downloaded: VTV
Downloaded: VUG
Downloaded: IWM
Downloaded: DVY
Downloaded: VEA
Downloaded: VGK
Downloaded: FEZ
Downloaded: EWJ
Downloaded: FXI
Downloaded: AFK
Downloaded: VWO
Downloaded: SHY
Downloaded: IEF
Downloaded: TLT
Downloaded: BND
Downloaded: TIP
Downloaded: VCSH
Downloaded: LQD
Downloaded: HYG
Downloaded: EMB
Downloaded: BWX
Downloaded: VNQ
Downloaded: RWX
Downloaded: IGF
Downloaded: GLD
Downloaded: SLV
Downloaded: USO
Downloaded: DBB
Downloaded: DBA
Downloaded: BTC-USD
Downloaded: ETH-USD


## FRED Data

In [11]:
# Directory where raw data will be stored
FRED_RAW_DIR = config.RAW_DATA_DIR / "fred"
FRED_RAW_DIR.mkdir(parents=True, exist_ok=True)

# load FRED API key
fred = Fred(api_key=config.fred_api_key)

# Load FRED dataset
FRED_SERIES = config.FRED_SERIES

# Download FRED data
for series_id in FRED_SERIES:
    download_fred_series(
        series_id=series_id,
        fred=fred,
        output_dir=FRED_RAW_DIR
    )

Downloaded: DFF
Downloaded: DGS3MO
Downloaded: DGS2
Downloaded: DGS10
Downloaded: DGS30
Downloaded: T10Y2Y
Downloaded: T10Y3M
Downloaded: THREEFYTP2
Downloaded: THREEFYTP5
Downloaded: THREEFYTP10
Downloaded: CPIAUCSL
Downloaded: T5YIFR
Downloaded: T5YIE
Downloaded: T10YIE
Downloaded: UNRATE
Downloaded: INDPRO
Downloaded: MTSDS133FMS
Downloaded: MTSO133FMS
Downloaded: MVGFD027MNFRBDAL
Downloaded: VIXCLS
Downloaded: BAA10Y
Downloaded: AAA10Y
Downloaded: DTWEXBGS
Downloaded: DCOILWTICO


## Fama-French Files

In [13]:
# Directory where raw data will be stored
FAMA_FRENCH_RAW_DIR = config.RAW_DATA_DIR / "fama_french"
FAMA_FRENCH_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Load Fama-French files info
FAMA_FRENCH_FILES = config.FAMA_FRENCH_FILES

# Download Fama-French files
for filename, url in FAMA_FRENCH_FILES.items():
    download_fama_french_files(
        url=url,
        filename=filename,
        output_dir=FAMA_FRENCH_RAW_DIR
    )

Downloaded: ['F-F_Research_Data_5_Factors_2x3_daily.csv']
Downloaded: ['F-F_Momentum_Factor_daily.csv']
